# Reproducing the paper*Serialization and the Character-System: Measuring Narrative Concentration in French Fiction (1800–1940)*This notebook walks through every result and figure in the paper, in the orderthey appear. It reads the derived tables in `data/derived/`, so it runs withoutthe raw pipeline output (~200 GB of `.book`/`.entities` files).Each section mirrors one script in `scripts/`; running the script from thecommand line produces the same numbers and writes them to `results/`.

In [ ]:
import sys, jsonfrom pathlib import PathROOT = Path.cwd().parentsys.path.insert(0, str(ROOT / "src"))import numpy as np, pandas as pdimport matplotlib.pyplot as plt%matplotlib inlinefrom charconc.corpus import attach_labels, load_author_labels, load_corpusfrom charconc.inference import (author_bootstrap_curve, author_level_test,                                author_permutation_test, clustered_ols)from charconc.metrics import gini, concentration_ratiolabels = load_author_labels()gallica = attach_labels(load_corpus("gallica"), labels)print(f"{len(gallica):,} novels, {gallica.author.nunique():,} authors, "      f"{gallica.year.min():.0f}–{gallica.year.max():.0f}")

## 1. What the index measuresThe Gini of PageRank ranges from 0 (every character equally central) to near 1(one character holds all centrality). Before looking at any trend, it helps tosee what a given value corresponds to.The interpretive anchor used in the paper: with a cast of twenty, the gapbetween a novel built on **two** co-equal protagonists and one built on a**single** protagonist is about 0.05 — which is also the size of the wholesecular movement we go on to trace.

In [ ]:
def synthetic_cast(n_leads, lead_share=0.5, total=20):    leads = [lead_share / n_leads] * n_leads    minors = [(1 - lead_share) / (total - n_leads)] * (total - n_leads)    return leads + minorsfor k in (1, 2, 3):    print(f"{k} protagonist(s) + {20-k} minor characters -> Gini {gini(synthetic_cast(k)):.3f}")

In [ ]:
# and on real novels, at the extremes of the rangeanchors = {"Les Habits noirs (Féval, 1863)": "habits-noirs",           "Madame Bovary (Flaubert)":       "madame-bovary",           "Pêcheur d'Islande (Loti, 1894)": "pecheur-dislande",           "Le Père Goriot (Balzac, 1839)":  "pere-goriot"}for title, key in anchors.items():    hit = gallica[gallica.file_name.str.lower().str.contains(key, na=False)]    if len(hit):        r = hit.iloc[0]        print(f"{title:34s} Gini {r.gini_pagerank:.2f}   top character {r.cr1_mentions:.0%} of mentions")

## 2. The trajectory (Section 5.2, Figure 3)Concentration does not evolve monotonically: it declines to a trough in the1850s–1860s and rises again through the turn of the century.

In [ ]:
yearly = gallica.groupby("year")["gini_pagerank"].mean()quad = np.polyfit(yearly.index.astype(float), yearly.values, 2)print(f"quadratic fit is {'convex (U-shaped)' if quad[0] > 0 else 'concave'}, "      f"vertex at {-quad[1] / (2 * quad[0]):.0f}")trough = gallica[gallica.year.between(1855, 1875)]peak   = gallica[gallica.year.between(1920, 1940)]print(f"trough mean {trough.gini_pagerank.mean():.3f} -> peak mean {peak.gini_pagerank.mean():.3f}")print(f"leading character: {trough.cr1_mentions.mean():.1%} -> {peak.cr1_mentions.mean():.1%} of mentions")

In [ ]:
# the effect is small beside the variation between individual novelssd = trough.gini_pagerank.std()shift = peak.gini_pagerank.mean() - trough.gini_pagerank.mean()print(f"historical shift {shift:.3f} vs within-period sd {sd:.3f}  ({sd/shift:.1f}x larger)")print(f"{(trough.gini_pagerank > peak.gini_pagerank.mean()).mean():.0%} of novels at the "      "trough are already above the peak mean")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))roll = yearly.rolling(15, center=True, min_periods=5).mean()ax.axvspan(1840, 1914, alpha=0.13, color="#c0392b")ax.plot(roll.index, roll.values, color="#1f2d5a", lw=2.4)ax.set_xlim(1800, 1940); ax.grid(alpha=.2)ax.set_xlabel("Publication year"); ax.set_ylabel("Gini of PageRank")ax.set_title("Shaded: age of the roman-feuilleton (1840–1914)", fontsize=10);

## 3. Period, genre and canon (Section 5.4, Figure 4)Estimated jointly, with errors clustered by author. Note that the genrecoefficients run **opposite** to what a genre-driven account would predict,and that the canon coefficient does not survive clustering.

In [ ]:
labelled = gallica[gallica.is_canon.notna()]model = clustered_ols(labelled, period_ref="[1860, 1880)")summary = pd.DataFrame({"coef": model.params, "p_clustered": model.pvalues}).drop("const")summary.round(4)

## 4. Author signatures (Section 5.5, Figure 5)Author identity explains a substantial but minority share of the variance. Theraw $R^2$ rises mechanically with the number of author dummies, so the adjustedfigure is the defensible one.

In [ ]:
import statsmodels.api as smrows = []for thr in (1, 3, 5, 10, 20):    counts = gallica.author.value_counts()    sub = gallica[gallica.author.isin(counts[counts >= thr].index)]    X = sm.add_constant(pd.get_dummies(sub.author, drop_first=True).astype(float))    fit = sm.OLS(sub.gini_pagerank.astype(float), X).fit()    rows.append({"min novels/author": thr, "authors": sub.author.nunique(),                 "R2": round(fit.rsquared, 3), "R2 adjusted": round(fit.rsquared_adj, 3)})pd.DataFrame(rows)

## 5. The serialization test (Section 5.6, Figure 6)The headline result. The label is a property of the **author**, so the novelsare not independent observations — hence three estimates that do not assumethey are.

In [ ]:
panel = gallica.dropna(subset=["serial"]).copy()panel["serial"] = panel.serial.astype(int)print(f"{(panel.serial==1).sum():,} novels by {panel[panel.serial==1].author.nunique()} feuilleton authors")print(f"{(panel.serial==0).sum():,} novels by {panel[panel.serial==0].author.nunique()} control authors")print("\n(a) author-level means")for k, v in author_level_test(panel).items():    print(f"    {k:24s} {v:.4f}" if isinstance(v, float) else f"    {k:24s} {v}")

In [ ]:
print("(b) OLS net of period, genre, canon — errors clustered by author")m = clustered_ols(panel, treatment="serial")ci = m.conf_int()print(f"    coef {m.params['serial']:+.4f}  "      f"CI [{ci.loc['serial',0]:+.4f}, {ci.loc['serial',1]:+.4f}]  p = {m.pvalues['serial']:.4f}")

In [ ]:
print("(c) permutation across authors (reduced draws for the notebook)")perm = author_permutation_test(panel, n_draws=5000)print(f"    observed {perm['observed']:+.4f}   p = {perm['p_value']:.4f}")print("    (the paper reports 100,000 draws: run scripts/04_serialization.py)")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))years = np.arange(1840, 1914)for value, colour, label in [(1, "#c0392b", "feuilleton authors"),                             (0, "#1f2d5a", "literary control")]:    mean, se = author_bootstrap_curve(panel[panel.serial == value], years, n_draws=200)    ax.plot(years, mean, color=colour, lw=2.4, label=label)    ax.fill_between(years, mean - se, mean + se, color=colour, alpha=.18)ax.set_ylim(.34, .50); ax.set_xlim(1840, 1914); ax.grid(alpha=.2)ax.set_xlabel("Publication year"); ax.set_ylabel("Gini of PageRank")ax.legend(loc="lower right");

## 6. What does *not* explain the post-1880 riseConcentration rises inside the serial group itself after 1880, faster than inthe rest of the corpus. Several natural candidates fail to account for it —reported here as negative results.

In [ ]:
serial = panel[panel.serial == 1]for label, lo, hi in [("1840–1880", 1840, 1880), ("1880–1914", 1880, 1914)]:    w = serial[serial.year.between(lo, hi)]    X = sm.add_constant(w[["year"]].astype(float))    fit = sm.OLS(w.gini_pagerank.astype(float), X).fit(        cov_type="cluster", cov_kwds={"groups": w.author})    print(f"{label}: slope {fit.params['year']:+.5f}/year, p = {fit.pvalues['year']:.3f}")

In [ ]:
# cast size cannot be the cause: the cutoff holds it fixedprint(f"{(gallica.n_characters >= gallica.n_characters.max()).mean():.1%} of novels "      "sit exactly at the twenty-character cutoff")# nor the recurring hero, once tested at the author levelfrom charconc.corpus import foldfrom scipy import statsheroes = [fold(a) for a in labels["recurring_hero"]["authors"]]agg = (serial.assign(hero=serial.author.isin(heroes).astype(int))             .groupby("author").agg(v=("gini_pagerank","mean"), hero=("hero","first")))a, b = agg[agg.hero==1].v, agg[agg.hero==0].vprint(f"series heroes {a.mean():.3f} (n={len(a)}) vs none {b.mean():.3f} (n={len(b)}), "      f"exact p = {stats.mannwhitneyu(a, b, alternative='two-sided').pvalue:.3f}")

## 7. Running everythingThe full pipeline, including the 100,000-draw permutation and the twelvespecifications, is reproduced by:```bashmake all          # or: bash run_all.sh```Results are written to `results/` as JSON and CSV; figures to `figures/`.